### packages

note: no cohere access...

In [1]:
import openai
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery


In [12]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [3]:
query = "Can I get a tablet?"
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [6]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [7]:
results = retrieve_data(query, k=20)

In [8]:
results

{'retrieved_context_ids': ['B07Q38R2XF',
  'B07L8HX46F',
  'B0BGHMDZHG',
  'B08Z3N1YTR',
  'B00Y1NGRM2',
  'B004M5H6D8',
  'B081122SFR',
  'B08DNC3L4M',
  'B08DMY7S84',
  'B00B3507Q8',
  'B07RW5XZSH',
  'B07ZNPPXN6',
  'B0829D5423',
  'B08C2DVS1V',
  'B07YQW55CN',
  'B0788JXFXB',
  'B07FSHNRXX',
  'B0128Z5BPI',
  'B07KFXQFQ6',
  'B08Y55D9JM'],
 'retrieved_context': ['ASURION 2 Year Tablet Accident Protection Plan ($30 - $39.99) NO ADDITIONAL COST: You pay $0 for repairs – parts, labor and shipping included. COVERAGE: Plan starts on the date of purchase. Drops, spills and cracked screens due to normal use are covered from day one. Malfunctions are covered after the manufacturer’s warranty. EASY CLAIMS PROCESS: File a claim anytime online at www.Asurion.com/Amazon or by phone. Most claims approved within minutes. We will send you an Amazon e-gift card for the purchase price of your covered product. In some cases, we will replace or repair it. EXPERT TECH HELP: Real experts are available 

### reranking

In [14]:
cohere_client = cohere.ClientV2()

In [24]:
import os
os.environ['CO_API_URL']

'https://api.cohere.com/'

In [25]:
to_rerank = results["retrieved_context"]

In [26]:
to_rerank

['ASURION 2 Year Tablet Accident Protection Plan ($30 - $39.99) NO ADDITIONAL COST: You pay $0 for repairs – parts, labor and shipping included. COVERAGE: Plan starts on the date of purchase. Drops, spills and cracked screens due to normal use are covered from day one. Malfunctions are covered after the manufacturer’s warranty. EASY CLAIMS PROCESS: File a claim anytime online at www.Asurion.com/Amazon or by phone. Most claims approved within minutes. We will send you an Amazon e-gift card for the purchase price of your covered product. In some cases, we will replace or repair it. EXPERT TECH HELP: Real experts are available 24/7 to help with set-up, connectivity issues, troubleshooting and much more. TERMS & DETAILS: More information about this protection plan is available within the “Product guides and documents” section. Simply click “User Guide” for more info. Asurion will also email your plan confirmation with Terms & Conditions to the address associated with your Amazon account wi

In [11]:
query

'Can I get a tablet?'

In [29]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

ApiError: headers: {'content-length': '134', 'content-type': 'text/html; charset=UTF-8', 'date': 'Fri, 10 Jul 2026 16:23:03 GMT', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'connection': 'close'}, status_code: 403, body: <!doctype html><meta charset="utf-8"><meta name=viewport content="width=device-width, initial-scale=1"><title>403</title>403 Forbidden